# SETUP

In [1]:
import io
import os       
import requests
import pandas as pd
import duckdb
import yfinance as yf
from tqdm.auto import tqdm
from dotenv import load_dotenv
from fredapi import Fred

C:\GitHub\rlp-jym-projects\forward-fundamentals-study\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# DATA INGESTION

In [2]:
PATH_MAIN = os.path.join(os.getcwd(), 'data')
PATH_HOLDINGS = f"{PATH_MAIN}/holdings"
PATH_PRICES = f"{PATH_MAIN}/prices"
PATH_CALENDARS = f"{PATH_MAIN}/calendars"

os.makedirs(PATH_MAIN, exist_ok=True)
os.makedirs(PATH_HOLDINGS, exist_ok=True)
os.makedirs(PATH_PRICES, exist_ok=True)
os.makedirs(PATH_CALENDARS, exist_ok=True)

ETFS = [
  'XLC', 'XLY', 'XLP', 'XLE', 'XLF', 'XLV', 'XLI', 'XLB', 'XLRE', 'XAR',
  'KBE', 'XBI', 'KCE', 'XHE', 'XHS', 'XHB', 'KIE', 'XME', 'XES', 'XOP',
  'XPH', 'KRE', 'XRT', 'XSD', 'XSW', 'XTL', 'XTN', 'XLK', 'XLU'
]

ETFS_TEST = ['XLE', 'XLU']

## Download official SPDR holdings

In [3]:
def download_holdings(tickers, path=PATH_HOLDINGS):
    for etf in tqdm(tickers):
        try:
            ssga_url = f'https://www.ssga.com/us/en/intermediary/library-content/products/fund-data/etfs/us/holdings-daily-us-en-{etf.lower()}.xlsx'
            headers = {"User-Agent": "Mozilla/5.0"}
            response = requests.get(ssga_url, headers=headers)
            response.raise_for_status()
            # Save and partially clean
            duckdb.sql(f"""
                copy (
                    select
                        '{etf}' as etf
                        , Name as name
                        , Ticker as ticker
                        , Weight::numeric as weight
                    from read_xlsx('{ssga_url}', range = 'A5:H')
                    where true
                        and SEDOL != '-'
                        and ticker != '-'
                ) to '{path}/{etf.lower()}_holdings.parquet'
            """)
        except Exception as e:
            print(f"{etf} error: {e}")
            continue
    # Save consolidated holdings
    duckdb.sql(f"""
        copy (
            select *
            from read_parquet("{path}/*.parquet", union_by_name=True)
        ) to "{PATH_MAIN}/holdings.parquet"
    """)

In [4]:
download_holdings(tickers=ETFS)

100%|██████████| 29/29 [00:56<00:00,  1.93s/it]


In [4]:
holdings = duckdb.sql(f"""
    select *
    from read_parquet("{PATH_MAIN}/holdings.parquet")
""").fetchdf()

In [5]:
print(holdings.info())

<class 'pandas.DataFrame'>
RangeIndex: 1769 entries, 0 to 1768
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   etf     1769 non-null   str    
 1   name    1769 non-null   str    
 2   ticker  1769 non-null   str    
 3   weight  1769 non-null   float64
dtypes: float64(1), str(3)
memory usage: 99.3 KB
None


In [14]:
print(holdings)

      etf                          name ticker  weight
0     KBE               BANCORP INC/THE   TBBK   1.121
1     KBE        EQUITABLE HOLDINGS INC    EQH   1.056
2     KBE        NICOLET BANKSHARES INC    NIC   1.052
3     KBE      COREBRIDGE FINANCIAL INC   CRBG   1.050
4     KBE       JACKSON FINANCIAL INC A    JXN   1.049
...   ...                           ...    ...     ...
1764  XTN  GENCO SHIPPING + TRADING LTD    GNK   1.303
1765  XTN       FTAI INFRASTRUCTURE INC    FIP   0.703
1766  XTN  COVENANT LOGISTICS GROUP INC   CVLG   0.675
1767  XTN         HEARTLAND EXPRESS INC   HTLD   0.627
1768  XTN        HERTZ GLOBAL HLDGS INC    HTZ   0.545

[1769 rows x 4 columns]


## Download market data from yFinance

In [6]:
def download_prices(tickers, path=PATH_PRICES, interval="1d", period="5y"):
    unique_tickers = tickers.ticker.unique()
    print("Removed duplicate names.")
    unique_tickers = [ticker.replace('.', '-') for ticker in unique_tickers]
    print("Fixed ticker names.")
    for ticker in tqdm(unique_tickers):
        try:
            df = yf.download(
                ticker
                , interval=interval
                , period=period
                , multi_level_index=False
                , progress=False
                , auto_adjust=True
            ).reset_index()
            # Save and partially clean
            duckdb.sql(f"""
                copy (
                    select
                        '{ticker}' as ticker
                        , Date::date as date
                        , Open as open
                        , High as high
                        , Low as low
                        , Close as close
                        , Volume as volume
                        , (close*volume)::bigint as value
                    from df
                ) to '{path}/{ticker.lower()}_prices.parquet'
            """)
        except Exception as e:
            print(f"{ticker} error: {e}")
            continue
    # Save consolidated prices
    duckdb.sql(f"""
        copy (
            select *
            from read_parquet("{path}/*.parquet", union_by_name=True)
        ) to "{PATH_MAIN}/prices.parquet"
    """)    

In [16]:
download_prices(holdings)

Removed duplicate names.
Fixed ticker names.


 34%|███▍      | 496/1445 [07:37<15:36,  1.01it/s]HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 2200963D"}}}
$2200963D: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['2200963D']: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
 34%|███▍      | 497/1445 [07:40<24:30,  1.55s/it]

2200963D error: Binder Error: Column "Date" referenced that exists in the SELECT clause - but this column cannot be referenced before it is defined


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 2200964D"}}}
$2200964D: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['2200964D']: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
 34%|███▍      | 498/1445 [07:43<30:43,  1.95s/it]

2200964D error: Binder Error: Column "Date" referenced that exists in the SELECT clause - but this column cannot be referenced before it is defined


 36%|███▌      | 522/1445 [08:04<17:31,  1.14s/it]$9667921D: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['9667921D']: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
 36%|███▌      | 523/1445 [08:06<22:21,  1.46s/it]

9667921D error: Binder Error: Column "Date" referenced that exists in the SELECT clause - but this column cannot be referenced before it is defined


100%|██████████| 1445/1445 [22:05<00:00,  1.09it/s]


In [7]:
prices = duckdb.sql(f"""
    select *
    from read_parquet("{PATH_MAIN}/prices.parquet")
""").fetchdf()

In [8]:
print(prices.info())

<class 'pandas.DataFrame'>
RangeIndex: 1734696 entries, 0 to 1734695
Data columns (total 8 columns):
 #   Column  Dtype         
---  ------  -----         
 0   ticker  str           
 1   date    datetime64[us]
 2   open    float64       
 3   high    float64       
 4   low     float64       
 5   close   float64       
 6   volume  int64         
 7   value   int64         
dtypes: datetime64[us](1), float64(4), int64(2), str(1)
memory usage: 111.6 MB
None


In [9]:
print(prices.head())

  ticker       date        open        high         low       close   volume  \
0      A 2021-07-15  143.167554  144.103733  142.684989  143.765945  1733600   
1      A 2021-07-16  144.123015  144.595934  143.099978  143.736954  2169800   
2      A 2021-07-19  142.511249  142.810437  141.825999  142.434036  1835100   
3      A 2021-07-20  143.273703  145.976073  142.675327  144.094070  2246000   
4      A 2021-07-21  144.277450  144.682804  142.453353  143.756271  2255700   

       value  
0  249232643  
1  311880442  
2  261380700  
3  323635282  
4  324271021  


## Download earnings calendar from yFinance

In [10]:
def download_calendars(tickers, path=PATH_CALENDARS):
    unique_tickers = tickers.ticker.unique()
    print("Removed duplicate names.")
    unique_tickers = [ticker.replace('.', '-') for ticker in unique_tickers]
    print("Fixed ticker names.")
    for ticker in tqdm(unique_tickers):
        try:
            df = yf.Ticker(ticker).earnings_dates.reset_index()
            # Save and partially clean
            duckdb.sql(f"""
                copy (
                    select
                        '{ticker}' as ticker
                        , "Earnings Date"::date as earnings_date
                        , "EPS Estimate" as eps_estimate
                        , "Reported EPS" as eps_actual
                        , "Surprise(%)" as surprise
                    from df
                ) to '{path}/{ticker.lower()}_calendars.parquet'
            """)
        except Exception as e:
            print(f"{ticker} error: {e}")
            continue
    # Save consolidated calendars
    duckdb.sql(f"""
        copy (
            select *
            from read_parquet("{path}/*.parquet", union_by_name=True)
        ) to "{PATH_MAIN}/calendars.parquet"
    """)    

In [22]:
download_calendars(holdings)

Removed duplicate names.
Fixed ticker names.


  1%|          | 12/1445 [00:13<27:35,  1.16s/it]

ESNT error: ['Earnings Date']


  4%|▎         | 54/1445 [00:55<23:37,  1.02s/it]

FRME error: ['Earnings Date']


...

 89%|████████▉ | 1291/1445 [22:27<02:41,  1.05s/it]

RBLX error: ['Earnings Date']


 93%|█████████▎| 1343/1445 [23:20<01:43,  1.02s/it]

DBX error: ['Earnings Date']


100%|██████████| 1445/1445 [25:04<00:00,  1.04s/it]


In [11]:
calendars = duckdb.sql(f"""
    select
        ticker
        , earnings_date
    from read_parquet("{PATH_MAIN}/calendars.parquet")
""").fetchdf()

In [12]:
print(calendars.info())

<class 'pandas.DataFrame'>
RangeIndex: 31967 entries, 0 to 31966
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   ticker         31967 non-null  str           
 1   earnings_date  31967 non-null  datetime64[us]
dtypes: datetime64[us](1), str(1)
memory usage: 607.6 KB
None


In [14]:
print(calendars.head())

  ticker earnings_date
0      A    2026-08-27
1      A    2026-05-28
2      A    2026-02-26
3      A    2025-11-25
4      A    2025-08-28


## Download sentiment proxies from FRED

In [15]:
load_dotenv()
fred_api_key = os.getenv('fred')
fred = Fred(api_key=fred_api_key)

# CBOE Volatility Index: VIX (VIXCLS)
vix = fred.get_series('VIXCLS')
vix.name = 'vix'
# ICE BofA AAA US Corporate Index Option-Adjusted Spread (BAMLC0A1CAAA)
aaa = fred.get_series('BAMLC0A1CAAA')
aaa.name = 'aaa'
# ICE BofA BBB US Corporate Index Option-Adjusted Spread (BAMLC0A4CBBB)
bbb = fred.get_series('BAMLC0A4CBBB')
bbb.name = 'bbb'

sentiment_proxies = pd.concat([vix, aaa, bbb], axis=1, sort=False).reset_index().to_parquet(f"{PATH_MAIN}/sentiment.parquet")

In [16]:
sentiment = duckdb.sql(f"""
    select
        index::date as date
        , vix
        , aaa
        , bbb
    from read_parquet("{PATH_MAIN}/sentiment.parquet")
""").fetchdf()

In [17]:
print(sentiment.info())

<class 'pandas.DataFrame'>
RangeIndex: 9542 entries, 0 to 9541
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    9542 non-null   datetime64[us]
 1   vix     9228 non-null   float64       
 2   aaa     785 non-null    float64       
 3   bbb     785 non-null    float64       
dtypes: datetime64[us](1), float64(3)
memory usage: 298.3 KB
None


In [18]:
print(sentiment.head())

        date    vix  aaa  bbb
0 1990-01-02  17.24  NaN  NaN
1 1990-01-03  18.19  NaN  NaN
2 1990-01-04  19.22  NaN  NaN
3 1990-01-05  20.11  NaN  NaN
4 1990-01-08  20.26  NaN  NaN


## Download official EDGAR CIK list

In [19]:
url = "https://www.sec.gov/files/company_tickers.json"
headers = {"User-Agent": "Your Name your.email@example.com"}

cik_list = requests.get(url, headers=headers).json()
cik_list = pd.DataFrame(cik_list).T

duckdb.sql(f"""
    copy (
        select
            'CIK' || lpad(cik_str::varchar, 10, '0') as cik
            , ticker
            , title as name
        from cik_list
    ) to '{PATH_MAIN}/ciks.parquet'
""")

In [20]:
ciks = duckdb.sql(f"""
    select *
    from read_parquet("{PATH_MAIN}/ciks.parquet")
""").fetchdf()

In [21]:
print(ciks.info())

<class 'pandas.DataFrame'>
RangeIndex: 9304 entries, 0 to 9303
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   cik     9304 non-null   str  
 1   ticker  9304 non-null   str  
 2   name    9304 non-null   str  
dtypes: str(3)
memory usage: 578.9 KB
None


## Download filing dates from EDGAR

In [47]:
holdings_extended = duckdb.sql(f"""
    select
        etf
        , cik
        , a.ticker as ticker
        , a.name as name
        , weight
    from holdings a
    join ciks b on a.ticker == b.ticker
""").fetchdf()

In [48]:
print(holdings_extended.info())

<class 'pandas.DataFrame'>
RangeIndex: 1758 entries, 0 to 1757
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   etf     1758 non-null   str    
 1   cik     1758 non-null   str    
 2   ticker  1758 non-null   str    
 3   name    1758 non-null   str    
 4   weight  1758 non-null   float64
dtypes: float64(1), str(4)
memory usage: 134.7 KB
None


In [49]:
print(holdings_extended.head())

   etf            cik ticker               name  weight
0  XSD  CIK0001045810   NVDA        NVIDIA CORP   2.475
1  XLK  CIK0000320193   AAPL          APPLE INC  12.793
2  XLC  CIK0001652044  GOOGL  ALPHABET INC CL A  10.526
3  XSW  CIK0000789019   MSFT     MICROSOFT CORP   0.692
4  XRT  CIK0001018724   AMZN     AMAZON.COM INC   1.406


In [88]:
holdings_extended = holdings_extended.drop_duplicates(subset=['ticker']).head(5)

headers = {"User-Agent": "YourName [email protected]"}

for idx, row in tqdm(holdings_extended.iterrows()):
    try:
        c = row["cik"]
        t = row["ticker"]
        url = f"https://data.sec.gov/api/xbrl/companyfacts/{c}.json"
        response = requests.get(url, headers=headers)
        data = response.json()
        keys = data["facts"]["us-gaap"]["Assets"]["units"]["USD"]
        df = pd.DataFrame(keys)
        duckdb.sql(f"""
            select
                '{t}' as ticker
                , filed
            from df
        """).show()
    except:
        print(f"{t} - {c} - no keys")

1it [00:00,  1.35it/s]

┌─────────┬────────────┐
│ ticker  │   filed    │
│ varchar │  varchar   │
├─────────┼────────────┤
│ NVDA    │ 2009-08-20 │
│ NVDA    │ 2009-11-19 │
│ NVDA    │ 2010-03-18 │
│ NVDA    │ 2009-08-20 │
│ NVDA    │ 2009-11-19 │
│ NVDA    │ 2010-03-18 │
│ NVDA    │ 2010-05-21 │
│ NVDA    │ 2010-08-30 │
│ NVDA    │ 2010-12-07 │
│ NVDA    │ 2011-03-16 │
│  ·      │     ·      │
│  ·      │     ·      │
│  ·      │     ·      │
│ NVDA    │ 2025-05-28 │
│ NVDA    │ 2025-08-27 │
│ NVDA    │ 2025-11-19 │
│ NVDA    │ 2026-02-25 │
│ NVDA    │ 2025-05-28 │
│ NVDA    │ 2025-08-27 │
│ NVDA    │ 2025-11-19 │
│ NVDA    │ 2026-02-25 │
│ NVDA    │ 2026-05-20 │
│ NVDA    │ 2026-05-20 │
└─────────┴────────────┘
  136 rows   2 columns
  (20 shown)           



2it [00:01,  1.42it/s]

┌─────────┬────────────┐
│ ticker  │   filed    │
│ varchar │  varchar   │
├─────────┼────────────┤
│ AAPL    │ 2009-07-22 │
│ AAPL    │ 2009-10-27 │
│ AAPL    │ 2010-01-25 │
│ AAPL    │ 2010-10-27 │
│ AAPL    │ 2009-07-22 │
│ AAPL    │ 2009-10-27 │
│ AAPL    │ 2010-01-25 │
│ AAPL    │ 2010-01-25 │
│ AAPL    │ 2010-04-21 │
│ AAPL    │ 2010-07-21 │
│  ·      │     ·      │
│  ·      │     ·      │
│  ·      │     ·      │
│ AAPL    │ 2025-08-01 │
│ AAPL    │ 2025-10-31 │
│ AAPL    │ 2025-01-31 │
│ AAPL    │ 2025-05-02 │
│ AAPL    │ 2025-08-01 │
│ AAPL    │ 2025-10-31 │
│ AAPL    │ 2026-01-30 │
│ AAPL    │ 2026-05-01 │
│ AAPL    │ 2026-01-30 │
│ AAPL    │ 2026-05-01 │
└─────────┴────────────┘
  144 rows   2 columns
  (20 shown)           



3it [00:01,  1.57it/s]

┌─────────┬────────────┐
│ ticker  │   filed    │
│ varchar │  varchar   │
├─────────┼────────────┤
│ GOOGL   │ 2015-10-29 │
│ GOOGL   │ 2016-02-11 │
│ GOOGL   │ 2016-05-03 │
│ GOOGL   │ 2015-10-29 │
│ GOOGL   │ 2016-02-11 │
│ GOOGL   │ 2016-05-03 │
│ GOOGL   │ 2016-05-03 │
│ GOOGL   │ 2016-08-04 │
│ GOOGL   │ 2016-11-03 │
│ GOOGL   │ 2017-02-03 │
│   ·     │     ·      │
│   ·     │     ·      │
│   ·     │     ·      │
│ GOOGL   │ 2025-04-25 │
│ GOOGL   │ 2025-07-24 │
│ GOOGL   │ 2025-10-30 │
│ GOOGL   │ 2026-02-05 │
│ GOOGL   │ 2025-04-25 │
│ GOOGL   │ 2025-07-24 │
│ GOOGL   │ 2025-10-30 │
│ GOOGL   │ 2026-02-05 │
│ GOOGL   │ 2026-04-30 │
│ GOOGL   │ 2026-04-30 │
└─────────┴────────────┘
  88 rows    2 columns
  (20 shown)           



4it [00:02,  1.49it/s]

┌─────────┬────────────┐
│ ticker  │   filed    │
│ varchar │  varchar   │
├─────────┼────────────┤
│ MSFT    │ 2009-10-23 │
│ MSFT    │ 2010-01-28 │
│ MSFT    │ 2010-04-22 │
│ MSFT    │ 2010-07-30 │
│ MSFT    │ 2009-10-23 │
│ MSFT    │ 2010-01-28 │
│ MSFT    │ 2010-04-22 │
│ MSFT    │ 2010-07-30 │
│ MSFT    │ 2010-10-28 │
│ MSFT    │ 2011-01-27 │
│  ·      │     ·      │
│  ·      │     ·      │
│  ·      │     ·      │
│ MSFT    │ 2024-10-30 │
│ MSFT    │ 2025-01-29 │
│ MSFT    │ 2025-04-30 │
│ MSFT    │ 2025-07-30 │
│ MSFT    │ 2025-10-29 │
│ MSFT    │ 2026-01-28 │
│ MSFT    │ 2026-04-29 │
│ MSFT    │ 2025-10-29 │
│ MSFT    │ 2026-01-28 │
│ MSFT    │ 2026-04-29 │
└─────────┴────────────┘
  140 rows   2 columns
  (20 shown)           



5it [00:03,  1.47it/s]

┌─────────┬────────────┐
│ ticker  │   filed    │
│ varchar │  varchar   │
├─────────┼────────────┤
│ AMZN    │ 2009-07-24 │
│ AMZN    │ 2009-10-23 │
│ AMZN    │ 2010-01-29 │
│ AMZN    │ 2009-07-24 │
│ AMZN    │ 2009-10-23 │
│ AMZN    │ 2010-01-29 │
│ AMZN    │ 2010-04-23 │
│ AMZN    │ 2010-07-23 │
│ AMZN    │ 2010-10-22 │
│ AMZN    │ 2011-01-28 │
│  ·      │     ·      │
│  ·      │     ·      │
│  ·      │     ·      │
│ AMZN    │ 2025-05-02 │
│ AMZN    │ 2025-08-01 │
│ AMZN    │ 2025-10-31 │
│ AMZN    │ 2026-02-06 │
│ AMZN    │ 2025-05-02 │
│ AMZN    │ 2025-08-01 │
│ AMZN    │ 2025-10-31 │
│ AMZN    │ 2026-02-06 │
│ AMZN    │ 2026-04-30 │
│ AMZN    │ 2026-04-30 │
└─────────┴────────────┘
  147 rows   2 columns
  (20 shown)           



# DATA CLEANING

# DATA PROCESSING

# MODELING

# CONCLUSION